# Pose Estimation com o MediaPipe

Neste notebook, vamos explorar a estimativa de pose humana (*pose estimation*) com o [MediaPipe](https://ai.google.dev/edge/mediapipe), o framework de visão computacional em tempo real do Google.

Diferente da detecção de objetos, que só devolve uma caixa delimitadora por pessoa, a estimativa de pose localiza o **esqueleto** de cada pessoa na imagem: um conjunto de pontos de referência (*landmarks*) nomeados, como ombros, cotovelos, punhos, quadris, joelhos e tornozelos. É esse nível de detalhe que viabiliza aplicações de hiperautomação como análise ergonômica de postura em linhas de produção, contagem de repetições em aplicativos de fitness, captura de movimento para animação e jogos, e reconhecimento de gestos.

Vamos cobrir dois casos de uso, ambos a partir do mesmo vídeo:
1. **Esqueleto nomeado**, extraindo e rotulando os pontos do corpo em um único quadro.
2. **Contagem de giros**, processando o vídeo inteiro para detectar quantas voltas completas a pessoa deu, usando a profundidade estimada pelo MediaPipe.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install mediapipe==1.0.1
    !pip install moviepy==2.2.1
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos em outros notebooks, vamos utilizar mais algumas.
* `mediapipe`: Framework do Google com o modelo de estimativa de pose (`PoseLandmarker`).
* `urllib`: Para download do modelo.
* `moviepy`: Para apresentar o vídeo processado.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from moviepy import VideoFileClip

import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline
from IPython.display import Video

## 1. Como Funciona a Estimativa de Pose

O `PoseLandmarker` do MediaPipe é baseado no **BlazePose**, uma arquitetura desenhada para rodar em tempo real até mesmo em CPU. O pipeline funciona em duas etapas:

1. Um detector localiza a região da imagem ocupada por uma pessoa (similar a uma detecção de objetos comum).
2. Um segundo modelo, especializado, recebe essa região recortada e prevê **33 pontos de referência** do corpo: rosto, ombros, cotovelos, punhos, mãos, quadris, joelhos, tornozelos e pés.

Para cada ponto, o modelo devolve duas representações complementares:

* `pose_landmarks`: coordenadas normalizadas (`x`, `y` entre 0 e 1, relativas à largura/altura da imagem), ideais para desenhar o esqueleto sobre a imagem original.
* `pose_world_landmarks`: coordenadas 3D em **metros**, relativas ao centro do quadril, estimadas a partir da imagem 2D. É essa profundidade (`z`) que vamos usar na Seção 3 para detectar giros, algo impossível de calcular só com as coordenadas normalizadas em pixels.

Assim como o restante do curso, o MediaPipe distribui seus modelos prontos, num formato de pacote `.task`, em variantes `lite`, `full` e `heavy`, trocando velocidade por precisão.

## 2. Esqueleto em um Único Quadro

Antes de processar o vídeo inteiro, vamos entender a API extraindo e nomeando o esqueleto de um único quadro.

### 2.1. Baixando o modelo

Assim como fizemos com o MobileNetV2 no notebook de classificação, baixamos o modelo, salvamos na pasta `modelos` e reaproveitamos o arquivo em execuções futuras. Usamos a variante `lite`, que já é suficiente para os pontos do corpo e roda bem mais rápido em CPU, importante quando vamos processar o vídeo inteiro na Seção 3.

In [ ]:
def baixar_arquivo_modelo(url: str, nome_arquivo: str) -> Path:
    # Pasta para armazenar os modelos
    MODELOS_DIR = Path('modelos')
    MODELOS_DIR.mkdir(exist_ok=True)

    path_modelo = MODELOS_DIR / nome_arquivo

    if not path_modelo.exists():
        print("Baixando arquivo...")
        urlretrieve(url, path_modelo)
        print("Download concluído!")
    else:
        print("Arquivo já existe.")

    return path_modelo

url = (
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
    "pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
)

modelo_path = baixar_arquivo_modelo(url, 'pose_landmarker_lite.task')

### 2.2. Carregando o Pose Landmarker

A API de Tasks do MediaPipe segue o mesmo padrão para todas as tarefas: `BaseOptions` aponta para o arquivo `.task` baixado, e `running_mode` define como o modelo vai processar as imagens. Aqui usamos `IMAGE`, para um quadro isolado; na Seção 3, ao processar o vídeo inteiro, trocamos para `VIDEO`, que aproveita a ordem temporal dos quadros para um rastreamento mais estável.

`num_poses=1` limita a detecção a uma pessoa por imagem, suficiente para o nosso vídeo.

In [ ]:
options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(modelo_path)),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_poses=1,
)
landmarker = mp_vision.PoseLandmarker.create_from_options(options)

print("Modelo carregado com sucesso!")

### 2.3. Carregando um quadro do vídeo

Assim como fizemos no notebook de rastreamento, usamos o `cv2.VideoCapture` para abrir o vídeo e capturar o primeiro quadro.

In [ ]:
video_path = 'imagens/05/7342766-uhd_2160_3840_25fps.mp4'

cap = cv2.VideoCapture(video_path)
ok, quadro = cap.read()
cap.release()

quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 10))
plt.imshow(quadro_rgb)
plt.axis('off')
plt.show()

### 2.4. Detectando os pontos do corpo

Diferente do OpenCV, que lê imagens como um array `numpy` puro, o MediaPipe espera um objeto `mp.Image`, em `RGB`. O método `detect()` devolve um `PoseLandmarkerResult`, com as duas listas de pontos descritas na Seção 1: `pose_landmarks` (uma lista por pessoa detectada, aqui só a primeira nos interessa) e `pose_world_landmarks`.

In [ ]:
mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=quadro_rgb)
resultado = landmarker.detect(mp_img)

print(f"{len(resultado.pose_landmarks)} pessoa(s) detectada(s)")
print(f"{len(resultado.pose_landmarks[0])} pontos de referência por pessoa")

### 2.5. Nomeando os Pontos do Corpo

Os 33 pontos vêm identificados pelo enum `PoseLandmark` (por exemplo, `PoseLandmark.LEFT_SHOULDER`), mas sempre em inglês. Criamos um dicionário só com os principais pontos do corpo, os mais úteis para aplicações de postura e movimento, traduzidos para português.

Também usamos `PoseLandmarksConnections.POSE_LANDMARKS`, a lista de conexões oficiais entre os pontos (por exemplo, cotovelo-punho), para desenhar o esqueleto completo, e não só os pontos soltos.

In [ ]:
PoseLandmark = mp_vision.PoseLandmark
CONEXOES = mp_vision.PoseLandmarksConnections.POSE_LANDMARKS

NOMES_PT = {
    PoseLandmark.NOSE: "Nariz",
    PoseLandmark.LEFT_SHOULDER: "Ombro Esq.",
    PoseLandmark.RIGHT_SHOULDER: "Ombro Dir.",
    PoseLandmark.LEFT_ELBOW: "Cotovelo Esq.",
    PoseLandmark.RIGHT_ELBOW: "Cotovelo Dir.",
    PoseLandmark.LEFT_WRIST: "Mão Esq.",
    PoseLandmark.RIGHT_WRIST: "Mão Dir.",
    PoseLandmark.LEFT_HIP: "Quadril Esq.",
    PoseLandmark.RIGHT_HIP: "Quadril Dir.",
    PoseLandmark.LEFT_KNEE: "Joelho Esq.",
    PoseLandmark.RIGHT_KNEE: "Joelho Dir.",
    PoseLandmark.LEFT_ANKLE: "Tornozelo Esq.",
    PoseLandmark.RIGHT_ANKLE: "Tornozelo Dir.",
}

def desenhar_esqueleto(
    img: np.ndarray,
    landmarks: list,
    cor: tuple = (0, 220, 0),
    rotular: bool = False
) -> np.ndarray:
    """ Desenha o esqueleto (pontos + conexões) sobre a imagem, opcionalmente rotulando os principais pontos """
    img = img.copy()
    altura, largura = img.shape[:2]

    diagonal = np.sqrt(largura**2 + altura**2)
    escala = diagonal / 2200.0

    espessura_linha = max(2, int(3 * escala))
    raio_ponto = max(3, int(5 * escala))
    raio_rotulo = max(5, int(8 * escala))

    fonte = max(0.5, 0.9 * escala)
    espessura_texto = max(1, int(2 * escala))

    desloc_x = int(12 * escala)
    desloc_y = int(12 * escala)

    # Converte as coordenadas normalizadas (0 a 1) para pixels
    pontos = [(int(lm.x * largura), int(lm.y * altura)) for lm in landmarks]

    for conexao in CONEXOES:
        cv2.line(img, pontos[conexao.start], pontos[conexao.end], cor, espessura_linha, cv2.LINE_AA)
    for x, y in pontos:
        cv2.circle(img, (x, y), raio_ponto, cor, -1, cv2.LINE_AA)

    if rotular:
        for indice, nome in NOMES_PT.items():
            x, y = pontos[indice]
            cv2.circle(img, (x, y), raio_rotulo, (0, 140, 255), -1, cv2.LINE_AA)
            cv2.putText(img, nome, (x + desloc_x, y - desloc_y), cv2.FONT_HERSHEY_SIMPLEX, fonte, (0, 140, 255), espessura_texto, cv2.LINE_AA)

    return img

### 2.6. Apresentando o Resultado

In [ ]:
landmarks = resultado.pose_landmarks[0]
quadro_anotado = desenhar_esqueleto(quadro, landmarks, rotular=True)

plt.figure(figsize=(8, 13))
plt.imshow(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

altura, largura = quadro.shape[:2]
for indice, nome in NOMES_PT.items():
    lm = landmarks[indice]
    print(f"{nome}: ({int(lm.x * largura)}, {int(lm.y * altura)}), confiança={lm.visibility:.2f}")

Repare como cada ponto nomeado cai exatamente sobre a articulação correspondente, mesmo com a pessoa em pose dinâmica, com um braço estendido e o corpo levemente rotacionado. É esse rastreamento ponto a ponto, quadro a quadro, que vamos usar na próxima seção para medir rotação do corpo.

## 3. Contando os Giros no Vídeo

O vídeo mostra uma pessoa dançando, dando diversos giros sobre o próprio corpo. Vamos usar o `pose_world_landmarks` para detectar automaticamente quantas voltas completas ela dá.

### 3.1. A Técnica: Orientação do Tronco no Espaço 3D

Só as coordenadas da imagem (`x, y`) não são suficientes para medir um giro, pois a maior parte da rotação acontece na profundidade. Por isso utilizamos os `pose_world_landmarks` do MediaPipe, que fornecem coordenadas tridimensionais do corpo.

A orientação do tronco é estimada combinando os vetores formados pelos ombros e quadris. Observando esse vetor no plano horizontal (`x-z`), é possível calcular um ângulo que gira junto com a pessoa.

Como ângulos variam entre -180° e +180°, existe uma descontinuidade quando esse limite é ultrapassado. Para manter a rotação contínua, calculamos a diferença angular entre quadros consecutivos usando uma normalização que sempre escolhe o menor caminho entre dois ângulos e acumulamos essas pequenas rotações ao longo do vídeo.

Para reduzir o efeito de ruídos, desfoque e pequenas falhas na estimativa da pose, aplicamos uma suavização exponencial na orientação calculada e descartamos variações incompatíveis com a velocidade máxima de rotação esperada para uma pessoa.

Por fim, sempre que a rotação acumulada ultrapassa aproximadamente 360°, contabilizamos um novo giro completo. Em essência, o algoritmo funciona como um giroscópio virtual, estimando continuamente a orientação do tronco e acumulando as rotações observadas ao longo do tempo.

### 3.2. Carregando o Vídeo e o Modelo em Modo `VIDEO`

Fechamos o `landmarker` da Seção 2 (em modo `IMAGE`) e criamos um novo em modo `VIDEO`. Nesse modo, cada chamada a `detect_for_video()` recebe também o timestamp (em milissegundos) do quadro, o que permite ao MediaPipe usar o resultado de quadros anteriores para estabilizar a detecção do quadro atual.

In [ ]:
landmarker.close()

options = mp_vision.PoseLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(modelo_path)),
    running_mode=mp_vision.RunningMode.VIDEO,
    num_poses=1,
)
landmarker = mp_vision.PoseLandmarker.create_from_options(options)

output_path = 'output/giros_detectados.mp4'

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_quadros = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolução: {largura}x{altura}, {fps:.1f} fps, {total_quadros} quadros")

### 3.3. Função de Apresentação

Para cada quadro, desenhamos o esqueleto (sem os rótulos da Seção 2, que ficariam poluídos em vídeo) e informações sobre os giros, num retângulo semitransparente no canto superior esquerdo.

In [ ]:
def desenhar_informacoes(
    quadro: np.ndarray,
    giros: int,
    giros_horarios: int,
    giros_antihorarios: int,
    progresso: float,
    sentido: str,
    pose_detectada: bool
) -> np.ndarray:
    """Desenha o contador e o progresso do giro."""

    altura, largura = quadro.shape[:2]

    # Calcula variáveis de escala
    diagonal = np.sqrt(largura**2 + altura**2)
    escala = diagonal / 2200.0
    escala = max(0.8, escala)
    fonte = max(0.8, 1.0 * escala)
    espessura = max(2, int(2 * escala))
    margem = int(25 * escala)
    altura_linha = int(45 * escala)
    largura_painel = int(550 * escala)
    altura_painel = int(240 * escala)

    overlay = quadro.copy()

    cv2.rectangle(overlay, (margem, margem), (margem + largura_painel, margem + altura_painel), (0, 0, 0), -1)

    quadro = cv2.addWeighted(overlay, 0.65, quadro, 0.35, 0)

    cor_status = (0, 255, 0) if pose_detectada else (0, 0, 255)
    texto_status = "Pose detectada" if pose_detectada else "Pose nao detectada"

    y = margem + altura_linha
    cv2.putText(quadro, f"Giros totais: {giros}", (margem + 15, y), cv2.FONT_HERSHEY_SIMPLEX, fonte, (255, 255, 255), espessura, cv2.LINE_AA)
    
    y += altura_linha
    cv2.putText(quadro, f"Horario: {giros_horarios}", (margem + 15, y), cv2.FONT_HERSHEY_SIMPLEX, fonte * 0.85, (255, 255, 255), espessura, cv2.LINE_AA)
    
    y += altura_linha
    cv2.putText(quadro, f"Anti-horario: {giros_antihorarios}", (margem + 15, y), cv2.FONT_HERSHEY_SIMPLEX, fonte * 0.85, (255, 255, 255), espessura, cv2.LINE_AA)
    
    y += altura_linha
    cv2.putText(quadro, f"Sentido atual: {sentido}", (margem + 15, y), cv2.FONT_HERSHEY_SIMPLEX, fonte * 0.85, (255, 255, 255), espessura, cv2.LINE_AA)
    
    y += altura_linha
    cv2.putText(quadro, texto_status, (margem + 15, y), cv2.FONT_HERSHEY_SIMPLEX, fonte * 0.8, cor_status, espessura, cv2.LINE_AA)

    barra_x = margem + largura_painel + int(30 * escala)
    barra_y = margem + int(40 * escala)

    margem_direita = int(largura * 0.03)
    barra_w = min(int(largura * 0.35), largura - barra_x - margem_direita)
    barra_h = int(45 * escala)

    cv2.rectangle(quadro, (barra_x, barra_y), (barra_x + barra_w, barra_y + barra_h), (100, 100, 100), espessura)

    preenchimento = int(barra_w * np.clip(progresso, 0.0, 1.0))

    cv2.rectangle(quadro, (barra_x, barra_y), (barra_x + preenchimento, barra_y + barra_h), (0, 255, 255), -1)
    cv2.putText(quadro, f"Progresso do giro: {progresso * 100:.0f}%", (barra_x, barra_y + barra_h + int(40 * escala)), cv2.FONT_HERSHEY_SIMPLEX, fonte * 0.8, (255, 255, 255), espessura, cv2.LINE_AA)

    return quadro

### 3.4. Processando o Vídeo, Quadro a Quadro

Para cada quadro, detectamos a pose e calculamos, de acordo com informações acumuladas, o giro.

In [ ]:
# Quanto maior, mais rápido o algoritmo reage ao movimento.
ALPHA = 0.80

# Rejeita somente saltos claramente implausíveis.
VELOCIDADE_ANGULAR_MAX = np.radians(1080) # 3 giros por segundo

# Qualidade mínima do vetor usado para estimar a orientação.
NORMA_MINIMA = 0.015

# Tolerância para considerar que um giro foi concluído.
# 0.90 significa que 324 graus já podem fechar um giro.
FRACAO_MINIMA_GIRO = 0.90

# Pequenas variações abaixo desse valor são tratadas como ruído.
VARIACAO_MINIMA = np.radians(1.0)

# Uma mudança de sentido maior que esse valor reinicia o giro em andamento.
TOLERANCIA_MUDANCA_SENTIDO = np.radians(25)

# Número máximo de frames sem pose antes de reiniciar a referência angular.
MAX_FRAMES_SEM_POSE = 10

In [ ]:
def wrap_to_pi(angulo: float) -> float:
    """Normaliza um ângulo para o intervalo entre -pi e +pi."""
    return (angulo + np.pi) % (2 * np.pi) - np.pi

In [ ]:
angulo_suavizado = None
angulo_anterior = None
timestamp_anterior = None

rotacao_giro_atual = 0.0
sentido_atual = 0

giros_horarios = 0
giros_antihorarios = 0
giros = 0

frames_sem_pose = 0
quadro_idx = 0

quadros_amostra = []
intervalo_amostra = max(total_quadros // 4, 1)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (largura, altura))

while True:
    ok, quadro = cap.read()
    if not ok:
        break

    quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=quadro_rgb)
    timestamp_ms = int(cap.get(cv2.CAP_PROP_POS_MSEC))
    resultado = landmarker.detect_for_video(mp_img, timestamp_ms)

    quadro_anotado = quadro.copy()
    pose_detectada = False

    if (resultado.pose_landmarks and resultado.pose_world_landmarks):
        pose_detectada = True
        frames_sem_pose = 0

        landmarks_2d = resultado.pose_landmarks[0]
        landmarks_3d = resultado.pose_world_landmarks[0]

        ombro_esq = landmarks_3d[PoseLandmark.LEFT_SHOULDER]
        ombro_dir = landmarks_3d[PoseLandmark.RIGHT_SHOULDER]

        quadril_esq = landmarks_3d[PoseLandmark.LEFT_HIP]
        quadril_dir = landmarks_3d[PoseLandmark.RIGHT_HIP]

        ombro_dx = ombro_dir.x - ombro_esq.x
        ombro_dz = ombro_dir.z - ombro_esq.z

        quadril_dx = quadril_dir.x - quadril_esq.x
        quadril_dz = quadril_dir.z - quadril_esq.z

        # Combinação de ombros e quadris para representar o tronco.
        dx = 0.60 * ombro_dx + 0.40 * quadril_dx
        dz = 0.60 * ombro_dz + 0.40 * quadril_dz

        norma = np.hypot(dx, dz)

        if norma >= NORMA_MINIMA:
            angulo_bruto = np.arctan2(dz, dx)

            if angulo_suavizado is None:
                angulo_suavizado = angulo_bruto
            else:
                erro_angular = wrap_to_pi(angulo_bruto - angulo_suavizado)
                angulo_suavizado = wrap_to_pi(angulo_suavizado + ALPHA * erro_angular)

            timestamp_atual = quadro_idx / fps

            if (angulo_anterior is not None and timestamp_anterior is not None):
                dt = timestamp_atual - timestamp_anterior

                variacao = wrap_to_pi(angulo_suavizado - angulo_anterior)
                velocidade_angular = (abs(variacao) / max(dt, 1e-6))
                variacao_valida = (abs(variacao) >= VARIACAO_MINIMA and velocidade_angular <= VELOCIDADE_ANGULAR_MAX)

                if variacao_valida:
                    sentido_variacao = (1 if variacao > 0 else -1)

                    if sentido_atual == 0:
                        sentido_atual = sentido_variacao

                    if sentido_variacao == sentido_atual:
                        rotacao_giro_atual += variacao

                    elif (abs(variacao) <= TOLERANCIA_MUDANCA_SENTIDO):
                        # Pequena correção contrária reduz parcialmente
                        # o progresso, sem cancelar imediatamente o giro.
                        rotacao_giro_atual += variacao

                    else:
                        # Mudança significativa de direção.
                        rotacao_giro_atual = variacao
                        sentido_atual = sentido_variacao

                    limite_giro = (2 * np.pi * FRACAO_MINIMA_GIRO)

                    if abs(rotacao_giro_atual) >= limite_giro:
                        if rotacao_giro_atual > 0:
                            giros_antihorarios += 1
                        else:
                            giros_horarios += 1

                        # Preserva eventual rotação excedente.
                        rotacao_excedente = (abs(rotacao_giro_atual) - 2 * np.pi)

                        if rotacao_excedente < 0:
                            rotacao_excedente = 0.0

                        rotacao_giro_atual = (sentido_atual * rotacao_excedente)

            angulo_anterior = angulo_suavizado
            timestamp_anterior = timestamp_atual

        quadro_anotado = desenhar_esqueleto(quadro_anotado, landmarks_2d)

    else:
        frames_sem_pose += 1

        if frames_sem_pose > MAX_FRAMES_SEM_POSE:
            angulo_suavizado = None
            angulo_anterior = None
            timestamp_anterior = None

    # Giros será a soma dos giros independentemente de sentido
    giros = giros_horarios + giros_antihorarios

    progresso = min(abs(rotacao_giro_atual) / (2 * np.pi), 1.0)

    if sentido_atual > 0:
        sentido_texto = "Anti-horario"
    elif sentido_atual < 0:
        sentido_texto = "Horario"
    else:
        sentido_texto = "Indefinido"

    quadro_anotado = desenhar_informacoes(
        quadro_anotado,
        giros,
        giros_horarios,
        giros_antihorarios,
        progresso,
        sentido_texto,
        pose_detectada
    )
    
    writer.write(quadro_anotado)

    if quadro_idx % intervalo_amostra == 0:
        quadros_amostra.append(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))

    quadro_idx += 1

# Finalização
cap.release()
writer.release()
landmarker.close()

print(
    f"Vídeo processado salvo em '{output_path}'.\n"
    f"Quadros processados: {quadro_idx}\n"
    f"Giros horários: {giros_horarios}\n"
    f"Giros anti-horários: {giros_antihorarios}\n"
    f"Total de giros: {giros}"
)

### 3.5. Apresentando o Resultado

Assim como no notebook de rastreamento, primeiro conferimos uma amostra dos quadros processados com o `matplotlib`, e depois tentamos exibir o vídeo completo.

In [ ]:
fig, eixos = plt.subplots(1, len(quadros_amostra), figsize=(3.2 * len(quadros_amostra), 6))

for eixo, quadro_amostra in zip(eixos, quadros_amostra):
    eixo.imshow(quadro_amostra)
    eixo.axis('off')

plt.tight_layout()
plt.show()

Se o seu navegador suportar o codec do vídeo gerado, ele será reproduzido logo abaixo. Caso contrário, abra o arquivo `output/giros_detectados.mp4` diretamente em um player de vídeo.

In [ ]:
def display_video(filename: str, **kwargs) -> None:
    """ Apresenta vídeo no notebook """
    clip = VideoFileClip(filename)
    html_embed = clip.display_in_notebook(**kwargs)
    # O moviepy cria um arquivo temporário. Só vamos apagar ele.
    Path("__temp__.mp4").unlink(missing_ok=True)
    return html_embed

In [ ]:
display_video(output_path, loop=1, height=600)

O contador no canto superior esquerdo acompanha a dança em tempo real, subindo cada vez que a pessoa completa mais uma volta. O mesmo princípio, comparar a orientação do corpo entre quadros a partir da profundidade estimada, vale para outras métricas de movimento derivadas de pose, como velocidade angular, amplitude de um gesto ou tempo de execução de um exercício.